# Industry-Wide Best Practices For Using Docker for Data Science

Run all cells inside a disposable Linux VM with a real Docker engine. No host folders, Docker Hub account, paid API or public registry is required. Install Python/ipykernel and (for the command notebook) pexpect and apache2-utils. Resources use unique labels and names; deletion checks identity before acting. Registry credentials are generated only in memory and temporary guest files. Original archive code remains verbatim in a nonexecuting appendix.

In [ ]:
import os, sys, json, uuid, tempfile, subprocess, shutil, platform, hashlib, time
from pathlib import Path
if platform.system() != 'Linux' or not shutil.which('docker'):
    raise RuntimeError('Use a disposable Linux VM with a real running Docker engine.')
endpoint = subprocess.check_output(['docker', 'context', 'inspect', '--format', '{{.Endpoints.docker.Host}}'], text=True).strip()
assert endpoint.startswith('unix://'), 'Only a local engine socket is permitted'
temporary = tempfile.TemporaryDirectory(prefix='tdb-docker-')
work = Path(temporary.name)
config_dir = work / 'docker-config'; config_dir.mkdir()
docker_env = dict(os.environ, DOCKER_HOST=endpoint, DOCKER_CONFIG=str(config_dir))
run_id = uuid.uuid4().hex[:12]; prefix = 'tdb20-' + run_id
owner_label = 'tdb.coding20.run=' + run_id
containers = {}; image_refs = {}; volumes = set(); build_logs = {}
def inspect_container(name):
    p = subprocess.run(['docker', 'inspect', name], env=docker_env, text=True, capture_output=True)
    return json.loads(p.stdout)[0] if p.returncode == 0 else None
def cli(args, *, quiet=False, check=True, input=None, timeout=1200):
    if args[0] in ['stop', 'rm']:
        for name in [x for x in args[1:] if not x.startswith('-')]:
            value = inspect_container(name)
            assert value and value['Config']['Labels'].get('tdb.coding20.run') == run_id
            assert containers.get(name) == value['Id'], 'Container identity changed'
    if args[0] == 'rmi':
        for name in [x for x in args[1:] if not x.startswith('-')]:
            assert name in image_refs
            actual = subprocess.check_output(['docker','image','inspect','--format','{{.Id}}',name],env=docker_env,text=True).strip()
            assert image_refs[name] == actual, 'Image reference changed'
    p = subprocess.run(['docker', *args], cwd=work, env=docker_env, text=True, input=input, capture_output=True, timeout=timeout)
    if check and p.returncode: raise RuntimeError(p.stderr)
    if not quiet:
        print(p.stdout, end='')
        if p.stderr: print('Docker stderr:', p.stderr.strip())
    return p
def image_info(ref): return json.loads(cli(['image','inspect',ref],quiet=True).stdout)[0]
def remember_image(ref):
    image_refs[ref] = image_info(ref)['Id']
    return ref
def create_container(suffix, *, running=True, image='ubuntu:24.04'):
    name = prefix + '-' + suffix
    assert inspect_container(name) is None
    args = ['run','-d'] if running else ['create']
    cli([*args,'--name',name,'--label',owner_label,image,'sleep','1200'],quiet=True)
    containers[name] = inspect_container(name)['Id']
    return name
def tag(source, target):
    assert target.startswith(prefix + '/') or target.startswith('localhost:')
    assert cli(['image','inspect',target],quiet=True,check=False).returncode != 0
    cli(['tag',source,target],quiet=True); remember_image(target)
    return target
def cleanup():
    for name in list(containers):
        if inspect_container(name): cli(['rm','-f','-v',name],quiet=True)
    for name in list(volumes):
        info = json.loads(cli(['volume','inspect',name],quiet=True).stdout)[0]
        assert info['Labels'].get('tdb.coding20.run') == run_id
        cli(['volume','rm',name],quiet=True)
    for ref in list(image_refs):
        if cli(['image','inspect',ref],quiet=True,check=False).returncode == 0:
            cli(['rmi',ref],quiet=True)
    temporary.cleanup()
engine = json.loads(cli(['info','--format','{{json .}}'],quiet=True).stdout)
assert engine['OSType'] == 'linux'
print('Python:', sys.version.split()[0], 'Actual Docker engine:', engine['ServerVersion'])
print('Linux kernel:', platform.release(), 'Architecture:', platform.machine())
print('Only uniquely labelled fixture containers, image references and volumes may be removed.')

In [ ]:
files = {'requirements.train.txt': 'numpy==2.2.4\nscipy==1.15.2\nscikit-learn==1.6.1\njoblib==1.4.2\nthreadpoolctl==3.6.0\npandas==2.2.3\n', 'my_script.py': 'import hashlib\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport pandas as pd\nimport seaborn as sns\n\ndata = pd.read_csv(Path(__file__).parent / "data/training.csv")\nassert data.shape == (6, 2)\nartifact = Path("/tmp/tdb-summary.png")\nsns.lineplot(data=data, x="feature", y="target")\nplt.tight_layout()\nplt.savefig(artifact)\nplt.close("all")\nassert artifact.read_bytes().startswith(b"\\x89PNG\\r\\n\\x1a\\n")\nprint(json.dumps({"rows": len(data), "feature_mean": float(data.feature.mean()),\n                  "target_mean": float(data.target.mean()), "plot_bytes": artifact.stat().st_size}, sort_keys=True))\n', 'requirements.plot.txt': 'pandas==2.2.3\nmatplotlib==3.10.1\nseaborn==0.13.2\n', 'Dockerfile.layers-before': 'FROM python:3.12-slim-bookworm\nWORKDIR /app\nRUN pip install --no-cache-dir pandas==2.2.3\nRUN pip install --no-cache-dir matplotlib==3.10.1\nRUN pip install --no-cache-dir seaborn==0.13.2\nCOPY my_script.py ./\nCOPY data/ ./data/\nCMD ["python", "my_script.py"]\n', 'predict.py': 'import json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\n\n# Load only the model created by our own build stage, never an untrusted upload.\nmodel = joblib.load(Path(__file__).parent / "models/regression.joblib")\ninputs = np.asarray([[6.0], [7.0]])\npredictions = model.predict(inputs)\nnp.testing.assert_allclose(predictions, [20.0, 23.0])\nprint(json.dumps({"inputs": inputs.ravel().tolist(), "predictions": predictions.round(8).tolist()}, sort_keys=True))\n', '.dockerignore': '__pycache__/\n*.ipynb\n*.log\n.env\n.git/\n', 'requirements.runtime.txt': 'numpy==2.2.4\nscipy==1.15.2\nscikit-learn==1.6.1\njoblib==1.4.2\nthreadpoolctl==3.6.0\n', 'train.py': 'import json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LinearRegression\n\ndata = pd.read_csv(Path(__file__).parent / "data/training.csv")\nmodel = LinearRegression().fit(data[["feature"]].to_numpy(), data.target.to_numpy())\nnp.testing.assert_allclose(model.coef_, [3.0])\nnp.testing.assert_allclose(model.intercept_, 2.0)\nnp.testing.assert_allclose(model.predict([[6.0], [7.0]]), [20.0, 23.0])\nPath("models").mkdir(exist_ok=True)\njoblib.dump(model, "models/regression.joblib")\nprint(json.dumps({"training_rows": len(data), "coefficient": float(model.coef_[0]),\n                  "intercept": float(model.intercept_)}, sort_keys=True))\n', 'Dockerfile.volume': 'FROM python:3.12-slim-bookworm\nWORKDIR /app\nCOPY main.py ./\nRUN mkdir -p /app/data\nVOLUME ["/app/data"]\nCMD ["python", "main.py"]\n', 'main.py': 'import json\nimport sys\nfrom pathlib import Path\n\npath = Path("/app/data/results.json")\nmode = sys.argv[1] if len(sys.argv) > 1 else "write"\nif mode == "write":\n    previous = json.loads(path.read_text()) if path.exists() else {"runs": 0}\n    value = {"runs": previous["runs"] + 1, "rows_processed": 6}\n    path.write_text(json.dumps(value, sort_keys=True))\nelif mode == "read":\n    value = json.loads(path.read_text())\nelse:\n    raise ValueError("Choose write or read")\nprint(json.dumps(value, sort_keys=True))\n', 'Dockerfile.multistage': 'FROM python:3.12-bookworm AS build\nWORKDIR /app\nCOPY requirements.train.txt ./requirements.txt\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY train.py ./\nCOPY data/ ./data/\nRUN python train.py\n\nFROM python:3.12-slim-bookworm AS runtime\nWORKDIR /app\nCOPY requirements.runtime.txt ./requirements.txt\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY --from=build /app/models ./models\nCOPY predict.py ./\nCMD ["python", "predict.py"]\n', 'Dockerfile.layers-after': 'FROM python:3.12-slim-bookworm\nWORKDIR /app\nCOPY requirements.plot.txt ./requirements.txt\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY my_script.py ./\nCOPY data/ ./data/\nCMD ["python", "my_script.py"]\n', 'data/training.csv': 'feature,target\n0,2\n1,5\n2,8\n3,11\n4,14\n5,17\n'}
context = work / 'context'; context.mkdir()
for name, text in files.items():
    target = context / name; target.parent.mkdir(parents=True, exist_ok=True); target.write_text(text)
def build(dockerfile, suffix, target=None):
    ref = prefix + '/' + suffix + ':test'
    args = ['build','--progress=plain','--label',owner_label,'-t',ref,'-f',str(context/dockerfile)]
    if target: args += ['--target',target]
    p = cli([*args,str(context)],quiet=True)
    build_logs[suffix] = p.stdout + p.stderr
    remember_image(ref)
    info = image_info(ref)
    print(json.dumps({'dockerfile':dockerfile,'image':ref,'layers':len(info['RootFS']['Layers']),'size_bytes':info['Size']},sort_keys=True))
    return ref
print('Supplied runnable build files:', len(files))

In [ ]:
before = build('Dockerfile.layers-before','layers-before')
before_result = json.loads(cli(['run','--rm','--label',owner_label,before],quiet=True).stdout)
print(json.dumps(before_result,sort_keys=True))

In [ ]:
after = build('Dockerfile.layers-after','layers-after')
after_result = json.loads(cli(['run','--rm','--label',owner_label,after],quiet=True).stdout)
for value in [before_result, after_result]:
    assert value['rows'] == 6 and value['feature_mean'] == 2.5 and value['target_mean'] == 9.5 and value['plot_bytes'] > 1000
assert len(image_info(after)['RootFS']['Layers']) < len(image_info(before)['RootFS']['Layers'])
cached = cli(['build','--progress=plain','--label',owner_label,'-t',after,'-f',str(context/'Dockerfile.layers-after'),str(context)],quiet=True)
remember_image(after)
assert 'CACHED' in cached.stdout + cached.stderr
print(json.dumps(after_result,sort_keys=True))
print('PASS: same data result, actual PNG output, fewer filesystem layers and real cached rebuild.')

In [ ]:
cli(['pull','python:3.12-slim-bookworm'],quiet=True)
official = image_info('python:3.12-slim-bookworm')
assert official['RepoDigests'] and official['Os'] == 'linux'
print('Resolved official image digest:', official['RepoDigests'][0])
print('A tag is mutable; record the resolved digest when reproducibility matters.')

In [ ]:
builder = build('Dockerfile.multistage','model-builder',target='build')
runtime = build('Dockerfile.multistage','model-runtime')
predicted = json.loads(cli(['run','--rm','--label',owner_label,runtime],quiet=True).stdout)
assert predicted == {'inputs':[6.0,7.0],'predictions':[20.0,23.0]}
probe = "import os,importlib.util; assert not os.path.exists('/app/train.py'); assert importlib.util.find_spec('pandas') is None; assert os.path.isfile('/app/models/regression.joblib'); print('Runtime contains Python, model and prediction code; no training script or pandas')"
print(cli(['run','--rm','--label',owner_label,runtime,'python','-c',probe],quiet=True).stdout,end='')
assert image_info(runtime)['Size'] < image_info(builder)['Size']
print(json.dumps(predicted,sort_keys=True))
print('PASS: real training, separate prediction image, independently expected 3*x+2 results and measured smaller runtime.')

In [ ]:
volume_image = build('Dockerfile.volume','volume')
volume_name = prefix + '-data'
cli(['volume','create','--label',owner_label,volume_name],quiet=True); volumes.add(volume_name)
first_name = prefix + '-writer'
assert inspect_container(first_name) is None
first = cli(['run','--name',first_name,'--label',owner_label,'--mount',f'type=volume,source={volume_name},target=/app/data',volume_image,'python','main.py','write'],quiet=True)
containers[first_name] = inspect_container(first_name)['Id']
assert json.loads(first.stdout) == {'rows_processed':6,'runs':1}
cli(['rm','-v',first_name],quiet=True)
assert cli(['volume','inspect',volume_name],quiet=True).returncode == 0
second = cli(['run','--rm','--label',owner_label,'--mount',f'type=volume,source={volume_name},target=/app/data',volume_image,'python','main.py','read'],quiet=True)
assert json.loads(second.stdout) == json.loads(first.stdout)
print(first.stdout.strip()); print(second.stdout.strip())
print('PASS: a named volume survives container removal, including docker rm -v, and a new container reads its unchanged data.')

In [ ]:
versioned = tag(after,prefix+'/analysis:1.0.2')
staging = tag(after,prefix+'/analysis:staging')
assert image_info(versioned)['Id'] == image_info(staging)['Id'] == image_info(after)['Id']
print('Version and environment tags resolve to the same tested image ID.')
print('Tags can be reassigned; an immutable digest identifies content.')

In [ ]:
assert len(files) == 13
assert predicted['predictions'] == [3*6+2,3*7+2]
assert all((context/name).read_text() == text for name,text in files.items())
print('PASS: every Dockerfile built and ran in the real engine, exact source files, actual plots, model predictions, named-volume persistence and image identity.')
cleanup()
print('Removed only this run’s labelled test resources; upstream image caches are retained.')

## Original source-code appendix

Every original block is retained verbatim, including generic syntax and incomplete Dockerfiles. Use the tested examples above.

```dockerfile
# Use the official Python image as the build image

FROM python:3.9


# Install the dependencies

RUN pip install pandas

RUN pip install matplotlib

RUN pip install seaborn


# Copy neccesary files

COPY my_script.py .

COPY data/ .


# Run the script

CMD ["python","my_script.py"]
```

```dockerfile
# Use the official Python image as the build image

FROM python:3.9


# Install the dependencies using requirements.txt

COPY my_script.py requirements.txt data/ .

RUN pip install --no-cache-dir -r requirements.txt


# Run the script

CMD ["python","my_script.py"]
```

```dockerfile
# Use the official Python image as the build image

FROM python:3.9 AS build

# Set the working directory

WORKDIR /app

# Copy the requirements.txt file

COPY requirements.txt ./

# Install the dependencies

RUN pip install --no-cache-dir -r requirements.txt

# Copy the application files

COPY . .

# Train the model

RUN python train.py

# Use the official Alpine Linux image as the runtime image

FROM alpine:3

# Set the working directory

WORKDIR /app

# Copy the model files from the build image

COPY --from=build /app/models /app/models

# Copy the requirements.txt file

COPY --from=build /app/requirements.txt /app

# Install the dependencies

RUN pip install --no-cache-dir -r requirements.txt

# Run the application

CMD ["python","predict.py"]
```

```dockerfile
# Use the official Python image as the base image

FROM python:3.9

# Set the working directory

WORKDIR /app

# Copy the requirements.txt file

COPY requirements.txt ./

# Install the dependencies

RUN pip install --no-cache-dir -r requirements.txt

# Copy the rest of the application files

COPY . .

# Create a directory for storing data

RUN mkdir /app/data



# Define a volume for the data directory

VOLUME /app/data

# Run the application

CMD ["python","main.py"]
```